# Module SC6 — Compare · Ensemble · Calibrate · Policy (10-class)

> **Run on Kaggle** — Settings: Accelerator **GPU T4 x2**, Internet **ON**, Environment **Always use latest**. Attach inputs listed below, then **Save Version (Save & Run All)** so the outputs can be chained into the next module.

**Inputs:** `thesis-common-10c`, Module-SC1 output, and the **notebook outputs of Modules SC2–SC5** (each `logits_<bb>.npz`). Chaining supports multiple notebook outputs — add all four.

**Outputs:** `comparison.csv`, `best_per_head.json`, `fusion_backbones.json`, `calibration.json` — `fusion_backbones.json` feeds Module SC7.

In [ ]:

import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| epochs:', T.EPOCHS)


In [ ]:

# ---- load every backbone's cached logits that is attached ----
named = {}
for nm in T.BACKBONES + ['fusion']:
    p = T.find_one(f'logits_{nm}.npz', required=False)
    if p: named[nm] = T.load_logits(p); print('loaded', nm, '<-', p)
assert named, "Attach at least one logits_*.npz (Module 2-5 output)."

def itemize(entry):
    (vlg,vy,vit),(tlg,ty,tit) = entry
    agg,uniq,idx = T.aggregate_by_item(tlg, tit); lab = T.item_labels(ty, tit, idx)
    vagg,vu,vidx = T.aggregate_by_item(vlg, vit); vlab = T.item_labels(vy, vit, vidx)
    return {'test':(agg,lab,np.array(uniq)), 'val':(vagg,vlab,np.array(vu))}
IT = {nm: itemize(e) for nm,e in named.items()}

### Per-backbone test metrics + per-head winners

In [ ]:

import pandas as pd
rows = {}
for nm,d in IT.items():
    agg,lab,it = d['test']
    m = T.item_metrics(agg, lab, it)
    rows[nm] = {'material_f1':m['material']['macro_f1'], 'material_acc':m['material']['acc'],
                'route_f1':m['route']['macro_f1'], 'threat_qwk':m['threat']['qwk'],
                'threat_mae':m['threat']['mae'], 'condition_auroc':m['condition']['auroc']}
comp = pd.DataFrame(rows).T; comp.to_csv('comparison.csv')
print(comp.round(3))
head_metric = {'material':'material_f1','route':'route_f1','threat':'threat_qwk','condition':'condition_auroc'}
best = {h: comp[col].astype(float).idxmax() for h,col in head_metric.items()}
json.dump(best, open('best_per_head.json','w'), indent=2)
fuse = sorted(set(b for b in best.values() if b in T.BACKBONES))
json.dump(fuse, open('fusion_backbones.json','w'), indent=2)
print('\nBEST PER HEAD:', best, '\nFUSION BACKBONES ->', fuse)

### Cheap ensembles (no training) — does combining backbones help each head?

In [ ]:

single = [nm for nm in IT if nm in T.BACKBONES]
ref = single[0]; agg_ref, lab_ref, it_ref = IT[ref]['test']
named_test = {nm: (IT[nm]['test'][0], IT[nm]['test'][2]) for nm in single}   # (logit_dict, items)
# mean ensemble + per-head-best ensemble
fused_mean = T.ensemble_logits(named_test, it_ref, mode='mean')
perhead = {h: best[h] for h in head_metric if best[h] in single}
fused_best = T.ensemble_logits(named_test, it_ref, per_head_backbone=perhead)
for label, fused in [('MEAN-ensemble', fused_mean), ('BEST-per-head ensemble', fused_best)]:
    m = T.item_metrics(fused, lab_ref, it_ref)
    print(f"{label:24s}  matF1={m['material']['macro_f1']:.3f}  routeF1={m['route']['macro_f1']:.3f}  "
          f"threatQWK={m['threat']['qwk']:.3f}  condAUROC={m['condition']['auroc']:.3f}")

### BEST-COMBO — per-head-best system (material/route=resnet50, threat=mobilenetv3, condition=mean). Saved as `best_combo.json` (does NOT overwrite the other outputs).

In [ ]:

# Assemble the per-head-best combo: each head uses the backbone that is finest for it,
# and condition uses the 4-backbone MEAN (a parameter-free ensemble that beat every single).
import numpy as np
singles = [nm for nm in IT if nm in T.BACKBONES]
# per-head winner among the attached SINGLE backbones (auto-includes SC12/SC13 if they win); condition = mean
COMBO = {h: comp.loc[singles, head_metric[h]].astype(float).idxmax() for h in ('material','route','threat')}
COMBO['condition'] = '__mean__'                        # 4-backbone mean (parameter-free, best for condition)
def _fb(nm): return nm if nm in IT else singles[0]
_, lab_ref, it_ref = IT[_fb(COMBO['material'])]['test']; it_ref = list(it_ref)
def _aligned(nm, head):
    agg, _, it = IT[nm]['test']; it = list(it)
    return agg[head][[it.index(x) for x in it_ref]]
combo_log = {}
for head, src in COMBO.items():
    if src == '__mean__':                                   # mean in PROB space (condition is binary)
        cp = np.clip(np.mean([T.sigmoid_np(_aligned(nm, head)) for nm in singles], axis=0), 1e-6, 1-1e-6)
        combo_log[head] = np.log(cp/(1-cp))                 # store as equiv. logit so item_metrics recovers it
    else:
        combo_log[head] = _aligned(_fb(src), head)
rep = T.full_report(combo_log, lab_ref, np.array(it_ref)); rep['_combo_sources'] = COMBO
json.dump(rep, open('best_combo.json','w'), indent=2)
s = rep['scalars']
print('BEST-COMBO (per-head-best):', COMBO)
print(f"  material  acc={s['material']['acc']:.3f}  macroF1={s['material']['macro_f1']:.3f}")
print(f"  route     macroF1={s['route']['macro_f1']:.3f}")
print(f"  threat    QWK={s['threat']['qwk']:.3f}  MAE={s['threat']['mae']:.3f}")
print(f"  condition AUROC={s['condition']['auroc']:.3f}")
print('saved best_combo.json (kept comparison.csv / best_per_head.json / fusion_backbones.json)')

### Temperature-scaling calibration — ECE before/after (fit on val, eval on test)

In [ ]:
# GUARDED temperature scaling: clamp T to [0.5,5] and KEEP the fitted T only if it improves VAL ECE,
# else fall back to T=1 (robust on the tiny 46-item val set, which otherwise overfits the temperature).
# All four heads (incl. the ordinal threat head) use a SINGLE scalar temperature under the same guard;
# a per-cut-point threat vector has 3x the parameters and over-fits val, which regresses threat ECE.
import torch
pick = comp[['material_f1','route_f1','threat_qwk']].astype(float).mean(1).idxmax()
print('calibrating:', pick)
(vagg,vlab,vit) = IT[pick]['val']; (tagg,tlab,tit) = IT[pick]['test']
def _ece_soft(z,y): return T.expected_calibration_error(T.softmax_np(z), y)
def _ece_bin(z,y):  p=T.sigmoid_np(z); return T.expected_calibration_error(np.stack([1-p,p],1), y)
def _ece_corn(z,y): return T.expected_calibration_error(T.corn_class_probs(torch.tensor(z)).numpy(), y)
def guard(Tfit, vz, vy, ece_fn):
    Tfit = float(np.clip(Tfit, 0.5, 5.0))
    return Tfit if ece_fn(vz/Tfit, vy) <= ece_fn(vz, vy) else 1.0
cal = {}
Tm = guard(T.fit_T_softmax(vagg['material'],vlab['material']), vagg['material'], vlab['material'], _ece_soft)
cal['material'] = {'T':Tm,'ece_before':_ece_soft(tagg['material'],tlab['material']),'ece_after':_ece_soft(tagg['material']/Tm,tlab['material'])}
Tr = guard(T.fit_T_softmax(vagg['route'],vlab['route']), vagg['route'], vlab['route'], _ece_soft)
cal['route'] = {'T':Tr,'ece_before':_ece_soft(tagg['route'],tlab['route']),'ece_after':_ece_soft(tagg['route']/Tr,tlab['route'])}
Tc = guard(T.fit_T_binary(vagg['condition'],vlab['condition']), vagg['condition'], vlab['condition'], _ece_bin)
cal['condition'] = {'T':Tc,'ece_before':_ece_bin(tagg['condition'],tlab['condition']),'ece_after':_ece_bin(tagg['condition']/Tc,tlab['condition'])}
Tt = guard(T.fit_T_corn(vagg['threat'],vlab['threat']), vagg['threat'], vlab['threat'], _ece_corn)
cal['threat'] = {'T':Tt,'ece_before':_ece_corn(tagg['threat'],tlab['threat']),'ece_after':_ece_corn(tagg['threat']/Tt,tlab['threat'])}
json.dump(cal, open('calibration.json','w'), indent=2)
for h,v in cal.items(): print(f"{h:10s} T={v['T']:.2f}  ECE {v['ece_before']:.3f} -> {v['ece_after']:.3f}")

### Policy / decision engine — hazard recall + decision cost (the deployment metric)

In [ ]:

mp = T.softmax_np(tagg['material']); cp = T.sigmoid_np(tagg['condition'])
# use CALIBRATED threat probs for the decision layer
tp = T.corn_class_probs(torch.tensor(tagg['threat']/np.array(cal['threat']['T']))).numpy()
routes, expl = T.apply_policy(mp, cp, tp)
rep = T.decision_report(routes, tlab['route'])
print('POLICY decision report:', {k:(round(v,3) if isinstance(v,float) else v) for k,v in rep.items()})
print('predicted route mix:', dict(pd.Series(routes).value_counts()))
# a few worked decisions
print('\nWORKED EXAMPLES:')
for i in range(min(6,len(routes))):
    print(f"  item{i}: mat={T.MATERIALS[mp[i].argmax()]:10s} cond={'Poor' if cp[i]>0.5 else 'Good'} "
          f"thr={int(tp[i].argmax())} -> {routes[i]}  ({expl[i][-1] if expl[i] else ''})")

### Confidence-aware reconciliation — cross-head consistency + the route head (interdependency)
Feeds the **calibrated** probabilities of all four heads into `T.decide`: material+condition imply an expected threat (reconciled conservatively with the threat head), and the **learned route head** is reconciled with the policy-rule route. Compares the plain policy vs this reconciled layer.

In [ ]:

import torch, pandas as pd
# calibrated probs of the calibrated backbone `pick` (uses the T's fit above)
mat_cal   = T.softmax_np(tagg['material']/cal['material']['T'])
route_cal = T.softmax_np(tagg['route']/cal['route']['T'])
cond_cal  = T.sigmoid_np(tagg['condition']/cal['condition']['T'])
thr_cal   = T.corn_class_probs(torch.tensor(tagg['threat']/np.array(cal['threat']['T']))).numpy()
# (A) baseline policy on calibrated probs (material+condition+threat only)
base_routes, _ = T.apply_policy(mat_cal, cond_cal, thr_cal)
base = T.decision_report(base_routes, tlab['route'])
# (B) confidence-aware reconciliation (adds route head + consistency + confidence gating)
dec   = T.decide(mat_cal, cond_cal, thr_cal, route_probs=route_cal)
recon = T.decision_report_v2(dec, tlab['route'], true_threat=tlab['threat'])
print('BASELINE policy   :', {k:(round(v,3) if isinstance(v,float) else v) for k,v in base.items()})
print('RECONCILED (v2)   :', {k:(round(v,3) if isinstance(v,float) else v) for k,v in recon.items()})
print('reconciled route mix:', dict(pd.Series(dec['route']).value_counts()))
json.dump({'baseline':base, 'reconciled':recon}, open('decision_compare.json','w'), indent=2, default=float)
print('\nCROSS-HEAD CONFLICTS surfaced (material/condition vs threat, or policy vs route head):')
shown = 0
for i in range(len(dec['route'])):
    f = dec['flags'][i]
    if (f['route_conflict'] or f['threat_conflict']) and shown < 8:
        print(f"  item{i}: route={dec['route'][i]:8s} thr={dec['threat'][i]} conf={dec['confidence'][i]:.2f}"
              f"  | {dec['expl'][i][-1]}")
        shown += 1